# Streamlit: Web Apps for Engineering Tools

## The Core Question

You wrote a 200-line Python script that computes airfoil performance. Your teammate asks: *"Can I run it without reading your code?"*

Streamlit turns a Python script into a **web app** in ~10 lines. Sliders, buttons, file uploaders — all in Python. No HTML, no JavaScript, no frontend framework.

## Why Streamlit?

- **Python only**: the entire UI is Python. No separate frontend.
- **Instant interactivity**: move a slider, the app re-runs and updates in <1s
- **Shareable**: `streamlit share` gives a public URL. `streamlit run` for local.
- **Perfect for internal tools**: calculators, data explorers, result viewers

## The Workflow

```
Write app.py (Python + st.* calls)
        |
        v
  streamlit run app.py
        |
        v
  Browser opens at http://localhost:8501
        |
        v
  User interacts -> app re-runs -> UI updates
```

## Setup

Streamlit is NOT run inside a notebook. This notebook **shows** the app code and explains how to run it.


In [ ]:
# Streamlit is a separate process. We show the app code here.
# To run: save the code below as app.py, then:
#   streamlit run app.py
#
# Install: pip install streamlit
import subprocess
result = subprocess.run(['pip', 'show', 'streamlit'], capture_output=True, text=True)
if result.returncode == 0:
    print('Streamlit is installed.')
    print(result.stdout.split('\n')[1])
else:
    print('Streamlit is NOT installed. Run: pip install streamlit')

## 1. The App: Airfoil Performance Calculator

**Concept**: The app has 3 sliders (Re, alpha, t/c), computes Cl and Cd analytically, and plots the results. Every time a slider moves, the entire script re-runs from top to bottom.

**Key insight**: Streamlit re-runs the whole script on every interaction. There are no "events" or "callbacks" — just top-to-bottom Python.


In [ ]:
# This is the FULL app.py content.
# Save it as a file and run: streamlit run app.py

app_code = '''
import streamlit as st
import numpy as np
import plotly.graph_objects as go

st.set_page_config(page_title="Airfoil Calculator", layout="wide")
st.title("Airfoil Performance Calculator")
st.markdown("Explore Cl and Cd as a function of Re, alpha, and thickness.")

# --- Sidebar: inputs ---
st.sidebar.header("Parameters")
Re = st.sidebar.slider("Reynolds number (x10^5)", 1, 20, 10) * 1e5
alpha = st.sidebar.slider("Angle of attack (deg)", 0, 20, 5)
t_over_c = st.sidebar.slider("Thickness ratio t/c (%)", 4, 20, 12)

# --- Compute (analytical stand-in for CFD) ---
a_rad = np.radians(alpha)
cl = 2 * np.pi * a_rad * (1 - 0.05 * np.log10(Re / 1e5))
cd = 0.008 + 0.00005 * t_over_c + 0.0001 * alpha**2 - 0.002 * np.log10(Re / 1e5)
cl_cd = cl / cd if cd > 0 else 0

# --- Main: metrics ---
col1, col2, col3 = st.columns(3)
col1.metric("Cl", f"{cl:.3f}")
col2.metric("Cd", f"{cd:.4f}")
col3.metric("Cl/Cd", f"{cl_cd:.1f}")

# --- Plot: Cl and Cd vs alpha for current Re ---
alpha_range = np.linspace(0, 20, 100)
cl_range = 2 * np.pi * np.radians(alpha_range) * (1 - 0.05 * np.log10(Re / 1e5))
cd_range = 0.008 + 0.00005 * t_over_c + 0.0001 * alpha_range**2 - 0.002 * np.log10(Re / 1e5)

fig = go.Figure()
fig.add_trace(go.Scatter(x=alpha_range, y=cl_range, mode="lines", name="Cl",
                          line=dict(color="blue", width=2)))
fig.add_trace(go.Scatter(x=alpha_range, y=cd_range, mode="lines", name="Cd",
                          line=dict(color="red", width=2)))
fig.add_vline(x=alpha, line_dash="dash", line_color="gray",
              annotation_text=f"alpha={alpha} deg")
fig.update_layout(title=f"Cl, Cd vs alpha (Re={Re:.0e}, t/c={t_over_c}%)",
                  xaxis_title="alpha (deg)", yaxis_title="Coefficient",
                  template="plotly_white", height=400)
st.plotly_chart(fig, use_container_width=True)

# --- Table: sweep ---
st.subheader("Parameter Sweep")
sweep_Re = [1e5, 5e5, 1e6, 5e6]
rows = []
for r in sweep_Re:
    c_l = 2 * np.pi * a_rad * (1 - 0.05 * np.log10(r / 1e5))
    c_d = 0.008 + 0.00005 * t_over_c + 0.0001 * alpha**2 - 0.002 * np.log10(r / 1e5)
    rows.append({"Re": f"{r:.0e}", "Cl": round(c_l, 3), "Cd": round(c_d, 4), "Cl/Cd": round(c_l/c_d, 1)})
st.dataframe(rows, use_container_width=True)
'''

# Show the code
st_code_display = app_code
print("=== app.py content ===")
print(st_code_display[:500] + "\n...")
print(f"\nTotal: {len(app_code)} characters, {app_code.count(chr(10))} lines")

**How it works**:
- `st.sidebar.slider()` creates a slider. When the user moves it, the variable `Re` gets a new value and the script re-runs.
- `st.metric()` displays a big number. Updates automatically on re-run.
- `st.plotly_chart()` embeds an interactive Plotly plot.
- `st.dataframe()` shows a table.

**No HTML. No CSS. No JavaScript. Just Python.**


## 2. Save and Run

**Concept**: The app is just a `.py` file. Save it, run `streamlit run`, and a browser tab opens.


In [ ]:
# Save the app to a file
with open('airfoil_app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

import os
print(f'Saved: airfoil_app.py ({os.path.getsize("airfoil_app.py")} bytes)')
print()
print('To run the app:')
print('  1. pip install streamlit')
print('  2. streamlit run airfoil_app.py')
print('  3. Browser opens at http://localhost:8501')
print()
print('To share with your team:')
print('  streamlit share airfoil_app.py')
print('  -> gives a public URL (https://....streamlit.app)')

## 3. Engineering Patterns

**Concept**: Streamlit apps follow a simple pattern: **inputs -> compute -> display**. Here are 5 patterns you'll use repeatedly:


In [ ]:
patterns = (
    "PATTERN 1: Parameter explorer\n"
    "  st.slider / st.selectbox / st.number_input\n"
    "  -> compute -> st.plotly_chart / st.dataframe\n\n"
    "PATTERN 2: File uploader\n"
    "  uploaded = st.file_uploader('Upload CSV')\n"
    "  df = pd.read_csv(uploaded)\n"
    "  -> plot / analyze / display\n\n"
    "PATTERN 3: Multi-page app\n"
    "  page = st.sidebar.radio('Page', ['Overview', 'Details', 'Export'])\n"
    "  if page == 'Overview': ...\n"
    "  elif page == 'Details': ...\n\n"
    "PATTERN 4: Button-triggered action\n"
    "  if st.button('Run optimization'):\n"
    "      result = optimize()\n"
    "      st.success(f'Done: {result}')\n\n"
    "PATTERN 5: Session state (persist across re-runs)\n"
    "  if 'count' not in st.session_state:\n"
    "      st.session_state.count = 0\n"
    "  st.session_state.count += 1\n"
    "  st.write(f'Runs: {st.session_state.count}')\n"
)
print(patterns)

**Engineering takeaway**: Your 500-line CFD post-processing script becomes a web app your team can use. They upload the result file, pick the case, and see the plots — without touching Python.


## Exercises

1. **Heat transfer calculator**: Build a Streamlit app for a fin heat-transfer calculator. Inputs: fin length, thickness, material (selectbox), ambient temp. Outputs: efficiency, heat dissipated, temperature profile plot.

2. **CFD result viewer**: Build an app where the user uploads a CSV (columns: `x_over_c, cp_upper, cp_lower`). The app plots Cp and computes Cl, Cd by integration.

3. **Multi-page**: Extend the airfoil app with 3 pages: (a) single-point calculator, (b) parameter sweep (table + plot), (c) export (download CSV of sweep results).

4. **Share it**: Run `streamlit share` and send the URL to a colleague. Can they use it without installing anything?

5. **Session state**: Add a "history" feature — every time the user changes parameters, append the result to a table that persists across slider moves.
